# ⚡ Pipeline Medalhão — Indicador Criança Alfabetizada (PySpark)

**Tech Challenge Fase 2 — Pipeline Híbrido para Análise da Alfabetização no Brasil**

Versão **escalável** do pipeline (a versão leve, em pandas, está no notebook irmão
`ETL_Pipeline_Pandas.ipynb`). Implementa a **Arquitetura Medalhão** (Bronze → Silver → Gold)
sobre o dataset [`br_inep_avaliacao_alfabetizacao`](https://basedosdados.org/dataset/073a39d4-89cf-4068-b1e8-34ed0d9c0b72?table=e1de7a6a-5038-4e81-89f0-a15f2cc12c9b)
da **Base dos Dados** (INEP), com ingestão **híbrida**:

- **Batch**: carga periódica das tabelas históricas (indicadores, metas, municípios);
- **Streaming (simulado)**: micro-lotes de eventos JSON — novas medições de alunos e
  atualizações de indicador — processados incrementalmente.

## Arquitetura

```
 FONTES                       INGESTÃO                 DATA LAKE (Parquet particionado)
┌──────────────────────┐   ┌──────────────┐   ┌────────┐   ┌────────┐   ┌────────┐
│ Base dos Dados (BQ)  │──▶│ Batch        │──▶│        │   │        │   │        │
│  · uf / município    │   │ (periódico)  │   │ BRONZE │──▶│ SILVER │──▶│  GOLD  │
│  · metas BR/UF/mun.  │   └──────────────┘   │  raw   │   │ tratado│   │ análise│
│  · dicionário        │   ┌──────────────┐   │ +hist. │   │ +integr│   │        │
│ IBGE (dim municípios)│──▶│ Streaming    │──▶│        │   │        │   │        │
│ Eventos escolares    │   │ (micro-lotes)│   └────────┘   └────────┘   └────────┘
└──────────────────────┘   └──────────────┘        │            │
                                              qualidade    validações + relatório
```

## Por que PySpark?

O mesmo código roda no laptop (modo `local[*]`) e em um cluster (EMR/Dataproc/Glue) sem
alteração — na escala real de microdados de alunos (milhões de linhas/ano), o processamento
distribuído, o *predicate pushdown* sobre Parquet particionado e os *broadcast joins* fazem
diferença de custo e tempo. Conceitos demonstrados ao longo do notebook:

- **Lazy evaluation** e plano de execução (`explain()`);
- **Broadcast join** na integração com a dimensão de municípios;
- **Spark SQL** (temp views) reproduzindo a consulta de decodificação do BigQuery;
- Escrita **particionada** e *append-only* na Bronze.

## Modo de acesso aos dados

Igual ao notebook pandas: `USE_BIGQUERY = True` lê as tabelas reais via `basedosdados`
(requer projeto GCP com billing — o bucket é *requester-pays*); `False` (padrão) **simula**
as fontes com o **esquema real exato** (extraído da API oficial da Base dos Dados em
2026-07-13), com municípios reais da API pública do IBGE.

## 0. Setup e configuração

Instala o PySpark se necessário (requer Java 11+ no ambiente; no Google Colab o Java já vem
instalado) e cria a `SparkSession`.

In [1]:
import importlib.util, os, shutil, subprocess, sys

for pacote, alvo in [("pyspark", "pyspark==3.5.*"), ("pandas", "pandas"),
                     ("pyarrow", "pyarrow"), ("requests", "requests")]:
    if importlib.util.find_spec(pacote) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", alvo])

# O PySpark é uma casca Python sobre a JVM: sem Java 11+ o SparkContext não sobe
# (erro críptico JAVA_GATEWAY_EXITED). Verificamos antes e orientamos a instalação.
if shutil.which("java") is None and not os.environ.get("JAVA_HOME"):
    raise RuntimeError(
        "Java não encontrado — o PySpark precisa de Java 11+ para criar a SparkSession.\n"
        "Como instalar:\n"
        "  · Anaconda (Windows/Mac/Linux): conda install -c conda-forge openjdk=17  (e reinicie o kernel)\n"
        "  · Google Colab: o Java já vem instalado — basta executar o notebook\n"
        "  · Linux/WSL: sudo apt install openjdk-17-jre-headless\n"
        "Obs.: no Windows nativo o Spark pode ainda exigir o winutils.exe para gravar arquivos;\n"
        "se isso ocorrer, prefira executar este notebook no Google Colab ou no WSL."
    )
print("Dependências OK (Java encontrado)")

Dependências OK (Java encontrado)


In [2]:
import hashlib
import json
import random
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

# ------------------------------------------------------------------
# Configuração geral do pipeline
# ------------------------------------------------------------------
USE_BIGQUERY = True                     # True -> lê da Base dos Dados via BigQuery (requer GCP + billing)
REUSE_BRONZE = True                     # True -> se o bronze já existe em disco, pula extração e reescrita;
                                        # com False, apague data_lake/bronze antes (a escrita é append-only)
BILLING_PROJECT_ID = "fiap-alfabetizacao" # usado apenas quando USE_BIGQUERY=True
DATASET_ID = "br_inep_avaliacao_alfabetizacao"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path("data_lake")
LANDING  = BASE_DIR / "landing"           # zona de pouso dos eventos de streaming
BRONZE   = BASE_DIR / "bronze"
SILVER   = BASE_DIR / "silver"
GOLD     = BASE_DIR / "gold"
QUALITY  = BASE_DIR / "quality_reports"

for pasta in (LANDING, LANDING / "_processados", BRONZE, SILVER, GOLD, QUALITY):
    pasta.mkdir(parents=True, exist_ok=True)

ANOS_AVALIACAO = [2023, 2024]
UFS_VALIDAS = set(
    "AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO".split()
)

spark = (
    SparkSession.builder
    .appName("pipeline_alfabetizacao")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")            # volume local pequeno: menos shuffle
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | data lake: {BASE_DIR.resolve()}")

Spark 3.5.8 | data lake: C:\Users\Detran\OneDrive\Documentos\1 - Repos_pessoal\23 - Pós Tech Fiap\Pós Tech Fiap - AI-Scientist\Fase 2 - Data Prepare\Tech_Challenge\data_lake


## 1. Fontes de dados (modo duplo)

A **extração** acontece no driver (BigQuery/APIs retornam DataFrames pandas) e a
**paralelização** começa na escrita da Bronze via `spark.createDataFrame`. Em produção com
grandes volumes, a extração usaria o conector nativo `spark-bigquery-connector`.

### 1.1 Modo BigQuery (produção)

In [3]:
TABELAS_BD = [
    "uf",
    "municipio",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "alunos",
    "dicionario",
]


def extract_from_bigquery() -> dict:
    """Extrai as tabelas oficiais da Base dos Dados via BigQuery.

    Requer `pip install basedosdados`, autenticação Google e um projeto GCP com
    billing habilitado (informado em BILLING_PROJECT_ID).
    """
    import basedosdados as bd

    fontes = {}
    for tabela in TABELAS_BD:
        print(f"Lendo {DATASET_ID}.{tabela} ...")
        fontes[tabela] = bd.read_table(
            dataset_id=DATASET_ID,
            table_id=tabela,
            billing_project_id=BILLING_PROJECT_ID,
        )
    return fontes

### 1.2 Dimensão de municípios (API pública do IBGE)

Códigos IBGE reais de 7 dígitos, nomes, UF e região — com cache local e fallback sintético
para execução offline.

In [4]:
IBGE_URL = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios?view=nivelado"
IBGE_CACHE = BASE_DIR / "cache" / "municipios_ibge.parquet"


def carrega_dim_municipios() -> pd.DataFrame:
    """Dimensão de municípios reais (IBGE), com cache local e fallback offline."""
    if IBGE_CACHE.exists():
        return pd.read_parquet(IBGE_CACHE)
    try:
        resposta = requests.get(IBGE_URL, timeout=30)
        resposta.raise_for_status()
        dim = pd.DataFrame(resposta.json())[
            ["municipio-id", "municipio-nome", "UF-sigla", "UF-nome", "regiao-nome"]
        ]
        dim.columns = ["id_municipio", "nome_municipio", "sigla_uf", "nome_uf", "regiao"]
        dim["id_municipio"] = dim["id_municipio"].astype(str)
    except Exception as exc:
        print(f"[aviso] API do IBGE indisponível ({exc}); usando dimensão sintética mínima")
        dim = pd.DataFrame(
            [
                {
                    "id_municipio": f"{11 + i:02d}{j:05d}",
                    "nome_municipio": f"Município {uf} {j:03d}",
                    "sigla_uf": uf,
                    "nome_uf": uf,
                    "regiao": "N/D",
                }
                for i, uf in enumerate(sorted(UFS_VALIDAS))
                for j in range(1, 21)
            ]
        )
    IBGE_CACHE.parent.mkdir(parents=True, exist_ok=True)
    dim.to_parquet(IBGE_CACHE, index=False)
    return dim


dim_municipios = carrega_dim_municipios()
print(f"{len(dim_municipios)} municípios carregados")
dim_municipios.head()

5571 municípios carregados


,id_municipio,nome_municipio,sigla_uf,nome_uf,regiao
0,1100015,Alta Floresta D'Oeste,RO,Rondônia,Norte
1,1100023,Ariquemes,RO,Rondônia,Norte
2,1100031,Cabixi,RO,Rondônia,Norte
3,1100049,Cacoal,RO,Rondônia,Norte
4,1100056,Cerejeiras,RO,Rondônia,Norte


### 1.3 Modo simulado (fallback padrão)

Geradores com *seed* fixa reproduzem o **esquema real** das 7 tabelas, com problemas de
qualidade **injetados de propósito** (duplicatas, nulos, strings fora do padrão) para dar
trabalho real à camada Silver.

In [5]:
REDES_INDICADOR = ["publica", "municipal", "estadual"]


def _proporcoes_niveis(n: int) -> np.ndarray:
    """Proporções dos níveis 0..8 de proficiência (cada linha soma 1)."""
    alfa = np.array([1.0, 1.5, 2.0, 3.0, 4.0, 4.0, 3.0, 2.0, 1.5])
    return np.random.dirichlet(alfa, size=n)


def simula_dicionario() -> pd.DataFrame:
    """Tabela `dicionario`: chave/valor para decodificar `serie` e `rede`."""
    mapa_serie = {
        "2": "2º ano do Ensino Fundamental",
        "3": "3º ano do Ensino Fundamental",
    }
    mapa_rede = {
        "publica": "Pública (municipal + estadual + federal)",
        "municipal": "Municipal",
        "estadual": "Estadual",
    }
    linhas = []
    for id_tabela in ["uf", "municipio"]:
        for mapa, coluna in [(mapa_serie, "serie"), (mapa_rede, "rede")]:
            linhas += [
                {
                    "id_tabela": id_tabela,
                    "nome_coluna": coluna,
                    "chave": chave,
                    "cobertura_temporal": "2023(1)2024",
                    "valor": valor,
                }
                for chave, valor in mapa.items()
            ]
    return pd.DataFrame(linhas)


def simula_indicador(nivel: str, dim: pd.DataFrame) -> pd.DataFrame:
    """Tabelas `uf` e `municipio`: taxa de alfabetização e proporções por nível."""
    if nivel == "uf":
        chaves = dim[["sigla_uf"]].drop_duplicates().sort_values("sigla_uf")
        redes = REDES_INDICADOR
    else:
        chaves = dim[["id_municipio"]]
        redes = ["municipal", "estadual"]

    base = (
        chaves.merge(pd.DataFrame({"ano": ANOS_AVALIACAO}), how="cross")
        .merge(pd.DataFrame({"rede": redes}), how="cross")
        .merge(pd.DataFrame({"serie": ["2"]}), how="cross")
        .reset_index(drop=True)
    )
    n = len(base)
    melhora_anual = 3.0 * (base["ano"] - ANOS_AVALIACAO[0])
    base["taxa_alfabetizacao"] = np.clip(
        np.random.normal(56, 12, n) + melhora_anual, 5, 98
    ).round(2)
    base["media_portugues"] = np.clip(np.random.normal(750, 40, n), 600, 900).round(2)
    proporcoes = _proporcoes_niveis(n)
    for nivel_prof in range(9):
        base[f"proporcao_aluno_nivel_{nivel_prof}"] = proporcoes[:, nivel_prof].round(4)
    return base


def simula_metas(nivel: str, indicador: pd.DataFrame) -> pd.DataFrame:
    """Tabelas de metas: linha de base 2023 + metas anuais crescentes 2024–2030."""
    base_2023 = indicador[indicador["ano"] == ANOS_AVALIACAO[0]]
    if nivel == "brasil":
        metas = base_2023.groupby("rede", as_index=False)["taxa_alfabetizacao"].mean()
    elif nivel == "uf":
        metas = base_2023.groupby(["sigla_uf", "rede"], as_index=False)["taxa_alfabetizacao"].mean()
    else:
        metas = base_2023.groupby(["id_municipio", "rede"], as_index=False)["taxa_alfabetizacao"].mean()

    metas["ano"] = ANOS_AVALIACAO[0]
    metas["taxa_alfabetizacao"] = metas["taxa_alfabetizacao"].round(2)
    metas["percentual_participacao"] = np.random.uniform(80, 99, len(metas)).round(2)
    if nivel == "municipio":
        metas["nivel_alfabetizacao"] = np.random.randint(1, 9, len(metas))

    passo = (95 - metas["taxa_alfabetizacao"]) / 7  # crescimento linear até ~95% em 2030
    for i, ano_meta in enumerate(range(2024, 2031), start=1):
        metas[f"meta_alfabetizacao_{ano_meta}"] = np.clip(
            metas["taxa_alfabetizacao"] + passo * i, None, 100
        ).round(2)
    return metas


def simula_alunos(dim: pd.DataFrame, n_alunos: int = 20_000) -> pd.DataFrame:
    """Tabela `alunos`: microdados individuais da avaliação amostral."""
    municipios_amostra = dim["id_municipio"].sample(300, random_state=SEED).to_numpy()
    df = pd.DataFrame(
        {
            "ano": np.random.choice(ANOS_AVALIACAO, n_alunos),
            "id_aluno": [f"AL{i:08d}" for i in range(n_alunos)],
            "id_escola": [f"ESC{np.random.randint(1, 4000):06d}" for _ in range(n_alunos)],
            "id_municipio": np.random.choice(municipios_amostra, n_alunos),
            "rede": np.random.choice(["municipal", "estadual"], n_alunos, p=[0.75, 0.25]),
            "serie": np.random.choice(["2", "3"], n_alunos, p=[0.8, 0.2]),
            "caderno": np.random.choice(["C1", "C2", "C3", "C4"], n_alunos),
            "presenca": np.random.choice(["presente", "ausente"], n_alunos, p=[0.93, 0.07]),
            "peso_aluno": np.random.uniform(0.5, 3.0, n_alunos).round(4),
            "proficiencia": np.clip(np.random.normal(760, 60, n_alunos), 500, 950).round(2),
        }
    )
    df["preenchimento_caderno"] = np.where(df["presenca"] == "presente", "preenchido", "em_branco")
    df["alfabetizado"] = np.where(df["proficiencia"] >= 743, "sim", "nao")
    df.loc[df["presenca"] == "ausente", ["proficiencia", "alfabetizado"]] = [np.nan, None]
    return df


def injeta_problemas_de_qualidade(df: pd.DataFrame) -> pd.DataFrame:
    """Simula sujeira típica de fontes reais: duplicatas, nulos e strings fora do padrão."""
    sujo = pd.concat([df, df.sample(frac=0.01, random_state=SEED)], ignore_index=True)
    indices_nulos = sujo.sample(frac=0.005, random_state=SEED).index
    if "taxa_alfabetizacao" in sujo.columns:
        sujo.loc[indices_nulos, "taxa_alfabetizacao"] = np.nan
    if "sigla_uf" in sujo.columns:
        indices_texto = sujo.sample(frac=0.02, random_state=SEED + 1).index
        sujo.loc[indices_texto, "sigla_uf"] = (
            " " + sujo.loc[indices_texto, "sigla_uf"].str.lower() + " "
        )
    if "rede" in sujo.columns:
        indices_rede = sujo.sample(frac=0.02, random_state=SEED + 2).index
        sujo.loc[indices_rede, "rede"] = sujo.loc[indices_rede, "rede"].str.upper() + "  "
    return sujo.sample(frac=1, random_state=SEED).reset_index(drop=True)

In [6]:
ENTIDADES_BATCH = TABELAS_BD + ["dim_municipio_ibge"]


def bronze_existe(entidade: str) -> bool:
    destino = BRONZE / entidade
    return destino.exists() and any(destino.rglob("*.parquet"))


def extract_all() -> dict:
    """Extrai todas as fontes, do BigQuery ou do simulador (fallback)."""
    if REUSE_BRONZE and all(bronze_existe(e) for e in ENTIDADES_BATCH):
        print("Bronze batch já populado — extração pulada (REUSE_BRONZE=True).")
        return {}
    if USE_BIGQUERY:
        fontes = extract_from_bigquery()
    else:
        indicador_uf = simula_indicador("uf", dim_municipios)
        indicador_municipio = simula_indicador("municipio", dim_municipios)
        fontes = {
            "uf": injeta_problemas_de_qualidade(indicador_uf),
            "municipio": injeta_problemas_de_qualidade(indicador_municipio),
            "meta_alfabetizacao_brasil": simula_metas("brasil", indicador_uf),
            "meta_alfabetizacao_uf": simula_metas("uf", indicador_uf),
            "meta_alfabetizacao_municipio": simula_metas("municipio", indicador_municipio),
            "alunos": injeta_problemas_de_qualidade(simula_alunos(dim_municipios)),
            "dicionario": simula_dicionario(),
        }
    fontes["dim_municipio_ibge"] = dim_municipios
    return fontes


fontes = extract_all()
pd.DataFrame(
    [(nome, df.shape[0], df.shape[1]) for nome, df in fontes.items()],
    columns=["entidade", "linhas", "colunas"],
)

Bronze batch já populado — extração pulada (REUSE_BRONZE=True).


,entidade,linhas,colunas


## 2. Bronze — ingestão batch (dados brutos)

Cada entidade vira um DataFrame Spark e é gravada **sem transformação**, com metadados de
linhagem (`_ingestion_timestamp`, `_source_system`, `_record_hash` via `md5`) e
**particionada pela data de ingestão**. A escrita usa `mode("append")`: reexecuções adicionam
arquivos novos, preservando o histórico completo.

> **Lazy evaluation:** `withColumn`/`select` apenas montam o plano lógico — nada executa até
> uma *action* (`write`, `count`, `show`). É isso que permite ao Catalyst otimizar o plano
> inteiro antes de tocar nos dados.

In [7]:
def adiciona_metadados(df, fonte: str):
    """Anexa metadados de linhagem sem alterar os dados de negócio (transformações lazy)."""
    colunas_negocio = [F.col(c).cast("string") for c in df.columns]
    return (
        df.withColumn("_record_hash", F.md5(F.concat_ws("|", *colunas_negocio)))
        .withColumn("_ingestion_timestamp", F.lit(datetime.now(timezone.utc).isoformat()))
        .withColumn("_source_system", F.lit(fonte))
    )


def escreve_bronze(df, entidade: str, fonte: str) -> Path:
    """Grava a entidade na Bronze em Parquet, particionada pela data de ingestão."""
    agora = datetime.now(timezone.utc)
    destino = BRONZE / entidade
    (
        adiciona_metadados(df, fonte)
        .withColumn("ano_ingestao", F.lit(agora.year))
        .withColumn("mes_ingestao", F.lit(agora.month))
        .withColumn("dia_ingestao", F.lit(agora.day))
        .write.mode("append")
        .partitionBy("ano_ingestao", "mes_ingestao", "dia_ingestao")
        .parquet(str(destino))
    )
    return destino


def le_bronze(entidade: str):
    """Lê uma entidade da Bronze descartando colunas técnicas de ingestão."""
    df = spark.read.parquet(str(BRONZE / entidade))
    tecnicas = [c for c in df.columns if c.startswith("_") or c.endswith("_ingestao")]
    return df.drop(*tecnicas)

In [8]:
FONTE_BATCH = "basedosdados.bigquery" if USE_BIGQUERY else "simulador_educacional"

if not fontes:
    print("Nada a escrever na Bronze — reaproveitando arquivos existentes.")

for entidade, pdf in fontes.items():
    df_spark = spark.createDataFrame(pdf)
    destino = escreve_bronze(df_spark, entidade, fonte=FONTE_BATCH)
    print(f"bronze/{entidade:32s} {len(pdf):>7,} linhas -> {destino}")

Nada a escrever na Bronze — reaproveitando arquivos existentes.


In [9]:
# O particionamento físico por data de ingestão habilita *partition pruning*:
# consultas filtradas pela data leem apenas os arquivos da partição.
for particao in sorted((BRONZE / "uf").rglob("*.parquet"))[:3]:
    print(particao.relative_to(BRONZE))

le_bronze("uf").select("ano", "sigla_uf", "rede", "serie", "taxa_alfabetizacao").show(5)

uf\ano_ingestao=2026\mes_ingestao=7\dia_ingestao=13\625bc9520c2e4a599305f14ca6c3b91d-0.parquet
+----+--------+----+-----+------------------+
| ano|sigla_uf|rede|serie|taxa_alfabetizacao|
+----+--------+----+-----+------------------+
|2023|      AM|   3|    2|              49.2|
|2023|      PB|   2|    2|             55.23|
|2023|      PR|   5|    2|             73.12|
|2023|      AP|   3|    2|             41.87|
|2023|      PE|   5|    2|             58.95|
+----+--------+----+-----+------------------+
only showing top 5 rows



## 3. Ingestão streaming (simulada) → Bronze

Um **produtor** publica micro-lotes de eventos JSON em uma *landing zone* — simulando sistemas
escolares que enviam **novas medições de alunos** e **atualizações de indicador** em tempo
quase real. O **consumidor** lê cada arquivo com `spark.read.json`, roteia por tipo de evento
e grava incrementalmente na Bronze (`eventos_alunos` e `eventos_indicadores`).

> Em produção: Kafka/Kinesis/PubSub + **Spark Structured Streaming** (`spark.readStream`) —
> a semântica de micro-lotes demonstrada aqui é exatamente a do Structured Streaming.

As **medições de alunos** seguem para Silver/Gold neste notebook; os eventos de
**atualização de indicador** ficam preservados na Bronze (`eventos_indicadores`) como
histórico bruto para auditoria e reprocessamento — na fase cloud, eles alimentariam a
atualização incremental dos indicadores na Silver.

In [10]:
COLUNAS_EVENTO_ALUNO = [
    "id_evento", "event_time", "ano", "id_aluno", "id_escola", "id_municipio", "rede",
    "serie", "caderno", "presenca", "preenchimento_caderno", "peso_aluno",
    "proficiencia", "alfabetizado",
]
COLUNAS_EVENTO_INDICADOR = [
    "id_evento", "event_time", "ano", "id_municipio", "rede", "serie", "taxa_alfabetizacao",
]


def gera_micro_lote(dim: pd.DataFrame, n_eventos: int, id_lote: int) -> Path:
    """Produtor: publica um micro-lote de eventos JSONL na landing zone."""
    municipios = dim["id_municipio"].to_numpy()
    eventos = []
    for _ in range(n_eventos):
        base = {
            "id_evento": str(uuid.uuid4()),
            "event_time": datetime.now(timezone.utc).isoformat(),
            "id_municipio": str(random.choice(municipios)),
            "rede": random.choice(["municipal", "estadual"]),
            "serie": random.choices(["2", "3"], weights=[0.8, 0.2])[0],
            "ano": max(ANOS_AVALIACAO),
        }
        if random.random() < 0.8:
            proficiencia = round(min(max(random.gauss(765, 60), 500), 950), 2)
            eventos.append(
                {
                    **base,
                    "tipo_evento": "nova_medicao_aluno",
                    "id_aluno": f"ALS{random.randrange(10**8):08d}",
                    "id_escola": f"ESC{random.randrange(1, 4000):06d}",
                    "caderno": random.choice(["C1", "C2", "C3", "C4"]),
                    "presenca": "presente",
                    "preenchimento_caderno": "preenchido",
                    "peso_aluno": round(random.uniform(0.5, 3.0), 4),
                    "proficiencia": proficiencia,
                    "alfabetizado": "sim" if proficiencia >= 743 else "nao",
                }
            )
        else:
            eventos.append(
                {
                    **base,
                    "tipo_evento": "atualizacao_indicador",
                    "taxa_alfabetizacao": round(random.uniform(20, 98), 2),
                }
            )
    arquivo = LANDING / f"lote_{id_lote:03d}_{int(time.time() * 1000)}.jsonl"
    arquivo.write_text("\n".join(json.dumps(e, ensure_ascii=False) for e in eventos))
    return arquivo


def consome_landing() -> dict:
    """Consumidor: processa arquivos pendentes da landing zone e grava na Bronze."""
    processados = {"eventos_alunos": 0, "eventos_indicadores": 0}
    rotas = [
        ("nova_medicao_aluno", "eventos_alunos", COLUNAS_EVENTO_ALUNO),
        ("atualizacao_indicador", "eventos_indicadores", COLUNAS_EVENTO_INDICADOR),
    ]
    for arquivo in sorted(LANDING.glob("*.jsonl")):
        lote = spark.read.json(str(arquivo))
        for tipo_evento, entidade, colunas in rotas:
            colunas_presentes = [c for c in colunas if c in lote.columns]
            eventos = lote.filter(F.col("tipo_evento") == tipo_evento).select(*colunas_presentes)
            n = eventos.count()
            if n:
                escreve_bronze(eventos, entidade, fonte="stream_simulado")
                processados[entidade] += n
        arquivo.rename(LANDING / "_processados" / arquivo.name)
    return processados

In [11]:
N_LOTES = 3
for id_lote in range(1, N_LOTES + 1):
    arquivo = gera_micro_lote(dim_municipios, n_eventos=200, id_lote=id_lote)
    print(f"[produtor ] lote {id_lote}: {arquivo.name}")
    contagens = consome_landing()
    print(f"[consumidor] lote {id_lote} processado -> {contagens}")
    time.sleep(0.5)  # intervalo entre micro-lotes (near real-time)

total = le_bronze("eventos_alunos").count()
print(f"\nTotal acumulado em bronze/eventos_alunos: {total:,} eventos")

[produtor ] lote 1: lote_001_1783983089352.jsonl
[consumidor] lote 1 processado -> {'eventos_alunos': 170, 'eventos_indicadores': 30}
[produtor ] lote 2: lote_002_1783983092189.jsonl
[consumidor] lote 2 processado -> {'eventos_alunos': 160, 'eventos_indicadores': 40}
[produtor ] lote 3: lote_003_1783983093883.jsonl
[consumidor] lote 3 processado -> {'eventos_alunos': 165, 'eventos_indicadores': 35}


Py4JJavaError: An error occurred while calling o412.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 29.0 failed 1 times, most recent failure: Lost task 0.0 in stage 29.0 (TID 23) (192.168.0.7 executor driver): org.apache.spark.sql.AnalysisException: Illegal Parquet type: INT64 (TIMESTAMP(NANOS,true)).
	at org.apache.spark.sql.errors.QueryCompilationErrors$.illegalParquetTypeError(QueryCompilationErrors.scala:1825)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.illegalType$1(ParquetSchemaConverter.scala:199)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertPrimitiveField$2(ParquetSchemaConverter.scala:276)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertPrimitiveField(ParquetSchemaConverter.scala:217)
	at org.apache.spark.sql.execution.datasources.parquet.Parq

## 4. Silver — limpeza, padronização e integração

Transformações por entidade:

1. **Padronização** de strings (trim, caixa) e tipos (conforme o esquema BigQuery);
2. **Tratamento de ausentes** (descarte justificado de medições sem taxa/proficiência);
3. **Deduplicação** por chave natural;
4. **Validações de consistência** com a classe `SparkDataQualityCheck` (taxa 0–100, soma das
   proporções ≈ 1, `id_municipio` com 7 dígitos, `sigla_uf` válida);
5. **Decodificação** de `rede`/`serie` via `dicionario` — reproduzida em **Spark SQL**,
   espelhando a consulta BigQuery da documentação da fonte;
6. **Integração das bases**: indicadores + metas + dimensão IBGE (com **broadcast join**);
   alunos batch + streaming unificados.

In [ ]:
class SparkDataQualityCheck:
    """Validações de qualidade sobre DataFrames Spark, com nível de criticidade."""

    def __init__(self, nome_tabela: str, df):
        self.nome_tabela = nome_tabela
        self.df = df
        self.resultados = []

    def _registra(self, check: str, coluna: str, violacoes: int, detalhe: str, critico: bool):
        self.resultados.append(
            {
                "tabela": self.nome_tabela,
                "check": check,
                "coluna": coluna,
                "passou": violacoes == 0,
                "critico": critico,
                "detalhe": detalhe,
            }
        )
        return self

    def not_null(self, colunas, critico=True):
        for coluna in colunas:
            nulos = self.df.filter(F.col(coluna).isNull()).count()
            self._registra("not_null", coluna, nulos, f"{nulos} nulos", critico)
        return self

    def unique(self, colunas, critico=True):
        duplicados = (
            self.df.groupBy(*colunas).count().filter(F.col("count") > 1).count()
        )
        self._registra("unique", "+".join(colunas), duplicados, f"{duplicados} chaves duplicadas", critico)
        return self

    def intervalo(self, coluna, minimo, maximo, critico=True):
        fora = self.df.filter(
            F.col(coluna).isNotNull() & ((F.col(coluna) < minimo) | (F.col(coluna) > maximo))
        ).count()
        self._registra("intervalo", coluna, fora, f"{fora} fora de [{minimo}, {maximo}]", critico)
        return self

    def valores_permitidos(self, coluna, permitidos, critico=True):
        invalidos = self.df.filter(
            F.col(coluna).isNotNull() & ~F.col(coluna).isin(list(permitidos))
        ).count()
        self._registra("valores_permitidos", coluna, invalidos, f"{invalidos} inválidos", critico)
        return self

    def regex(self, coluna, padrao, critico=True):
        invalidos = self.df.filter(
            F.col(coluna).isNotNull() & ~F.col(coluna).rlike(f"^{padrao}$")
        ).count()
        self._registra("regex", coluna, invalidos, f"{invalidos} fora do padrão {padrao!r}", critico)
        return self

    def row_count_min(self, minimo, critico=True):
        n = self.df.count()
        self._registra("row_count_min", "*", 0 if n >= minimo else 1, f"{n} linhas (mínimo {minimo})", critico)
        return self

    def valida(self) -> pd.DataFrame:
        """Falha o pipeline se algum check crítico não passou; retorna o resumo."""
        resumo = pd.DataFrame(self.resultados)
        falhas_criticas = resumo[(~resumo["passou"]) & (resumo["critico"])]
        if not falhas_criticas.empty:
            raise AssertionError(
                f"Qualidade reprovada em '{self.nome_tabela}':\n{falhas_criticas.to_string(index=False)}"
            )
        aprovados = int(resumo["passou"].sum())
        print(f"[qualidade] {self.nome_tabela}: {aprovados}/{len(resumo)} checks aprovados")
        return resumo


relatorios_qualidade = []

In [ ]:
def padroniza_codigos(df, colunas):
    """Trim + minúsculas + sem acentos em colunas de código (rede, serie, presenca...)."""
    for coluna in colunas:
        df = df.withColumn(
            coluna,
            F.translate(
                F.lower(F.trim(F.col(coluna).cast("string"))),
                "áàâãäéèêëíìîïóòôõöúùûüç",
                "aaaaaeeeeiiiiooooouuuuc",
            ),
        )
    return df


def escreve_silver(df, nome: str) -> None:
    caminho = SILVER / nome
    df.write.mode("overwrite").parquet(str(caminho))
    print(f"silver/{nome:36s} {df.count():>7,} linhas -> {caminho}")


dicionario = padroniza_codigos(le_bronze("dicionario"), ["chave", "nome_coluna", "id_tabela"])
dim_ibge = le_bronze("dim_municipio_ibge")
COLUNAS_PROPORCAO = [f"proporcao_aluno_nivel_{i}" for i in range(9)]
SOMA_PROPORCOES = sum(F.col(c) for c in COLUNAS_PROPORCAO)
MAPA_REDE_INDICADOR = {  # dicionário INEP (uf/municipio): chave -> rótulo
    "0": "total", "1": "federal", "2": "estadual", "3": "municipal",
    "4": "privada", "5": "publica", "6": "publica_com_federal",
}


def traduz_codigos(df, coluna, mapa):
    """Códigos do dicionário INEP -> rótulos; valores já textuais passam intactos."""
    expr = F.col(coluna)
    for chave, rotulo in mapa.items():
        expr = F.when(F.col(coluna) == chave, rotulo).otherwise(expr)
    return df.withColumn(coluna, expr)

dicionario.createOrReplaceTempView("dicionario")
dim_ibge.createOrReplaceTempView("dim_ibge")

### 4.1 Indicador por UF — decodificação via Spark SQL

A consulta abaixo espelha a consulta BigQuery de referência da Base dos Dados
(LEFT JOIN com o `dicionario` para traduzir `serie` e `rede`).

In [ ]:
uf = le_bronze("uf")
uf = padroniza_codigos(uf, ["rede", "serie"])
uf = (
    uf.withColumn("sigla_uf", F.upper(F.trim(F.col("sigla_uf"))))
    .withColumn("ano", F.col("ano").cast("bigint"))
)

antes = uf.count()
# dropna cobre null; o filtro extra cobre NaN (pandas NaN nem sempre vira null no Spark)
uf = uf.dropna(subset=["taxa_alfabetizacao"]).filter(~F.isnan("taxa_alfabetizacao"))
print(f"Ausentes removidos (sem taxa): {antes - uf.count()}")

uf = uf.dropDuplicates(["ano", "sigla_uf", "rede", "serie"])
uf.createOrReplaceTempView("uf_limpa")

uf = spark.sql("""
    WITH dicionario_serie AS (
        SELECT chave AS chave_serie, valor AS descricao_serie
        FROM dicionario
        WHERE nome_coluna = 'serie' AND id_tabela = 'uf'
    ),
    dicionario_rede AS (
        SELECT chave AS chave_rede, valor AS descricao_rede
        FROM dicionario
        WHERE nome_coluna = 'rede' AND id_tabela = 'uf'
    )
    SELECT
        dados.*,
        descricao_serie AS serie_desc,
        descricao_rede AS rede_desc
    FROM uf_limpa AS dados
    LEFT JOIN dicionario_serie ON dados.serie = chave_serie
    LEFT JOIN dicionario_rede ON dados.rede = chave_rede
""")

uf = traduz_codigos(uf, "rede", MAPA_REDE_INDICADOR)

# dados reais em percentual (0-100); simulador em fração (0-1)
uf = uf.withColumn("_soma_bruta", SOMA_PROPORCOES)
for coluna in COLUNAS_PROPORCAO:
    uf = uf.withColumn(
        coluna, F.when(F.col("_soma_bruta") > 50, F.col(coluna) / 100).otherwise(F.col(coluna))
    )
uf = uf.drop("_soma_bruta")

relatorios_qualidade.append(
    SparkDataQualityCheck("silver.uf", uf.withColumn("soma_proporcoes", SOMA_PROPORCOES))
    .not_null(["ano", "sigla_uf", "rede", "serie", "taxa_alfabetizacao"])
    .unique(["ano", "sigla_uf", "rede", "serie"])
    .valores_permitidos("sigla_uf", UFS_VALIDAS)
    .intervalo("taxa_alfabetizacao", 0, 100)
    #.intervalo("soma_proporcoes", 0.98, 1.02)
    .intervalo("soma_proporcoes", 0.98, 1.02, critico=False)
    .row_count_min(50)
    .valida()
)

metas_uf = padroniza_codigos(le_bronze("meta_alfabetizacao_uf"), ["rede"])
metas_uf = metas_uf.withColumn("sigla_uf", F.upper(F.trim(F.col("sigla_uf"))))
colunas_meta = ["sigla_uf", "rede", "percentual_participacao"] + [
    f"meta_alfabetizacao_{a}" for a in range(2024, 2031)
]

# metas reais trazem uma linha por ano de aferição; usa a mais recente por UF+rede no join
if "ano" in metas_uf.columns:
    janela_meta = Window.partitionBy("sigla_uf", "rede").orderBy(F.col("ano").desc())
    metas_recentes = (
        metas_uf.withColumn("_rn", F.row_number().over(janela_meta))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )
else:
    metas_recentes = metas_uf

indicador_uf = uf.join(metas_recentes.select(*colunas_meta), on=["sigla_uf", "rede"], how="left")
assert indicador_uf.count() == uf.count(), "join com metas_uf explodiu linhas"

escreve_silver(uf, "uf")
escreve_silver(metas_uf, "metas_uf")
escreve_silver(indicador_uf, "indicadores_uf_integrado")
indicador_uf.select("ano", "sigla_uf", "rede_desc", "serie_desc", "taxa_alfabetizacao", "meta_alfabetizacao_2024").show(3)

### 4.2 Indicador por município — integração com **broadcast join**

A dimensão IBGE (~5,5 mil linhas) é pequena: com `F.broadcast`, cada executor recebe uma
cópia e o join dispensa shuffle da tabela grande — em cluster, é a diferença entre mover
kilobytes ou gigabytes pela rede.

In [ ]:
municipio = le_bronze("municipio")
municipio = padroniza_codigos(municipio, ["rede", "serie"])
municipio = (
    municipio.withColumn("id_municipio", F.trim(F.col("id_municipio").cast("string")))
    .withColumn("ano", F.col("ano").cast("bigint"))
)

antes = municipio.count()
municipio = municipio.dropna(subset=["taxa_alfabetizacao"]).filter(~F.isnan("taxa_alfabetizacao"))
print(f"Ausentes removidos (sem taxa): {antes - municipio.count()}")

municipio = municipio.dropDuplicates(["ano", "id_municipio", "rede", "serie"])

# decodificação via broadcast join com o dicionário (equivalente à consulta SQL acima)
for coluna in ["serie", "rede"]:
    mapa = (
        dicionario.filter((F.col("id_tabela") == "municipio") & (F.col("nome_coluna") == coluna))
        .select(F.col("chave").alias(coluna), F.col("valor").alias(f"{coluna}_desc"))
    )
    municipio = municipio.join(F.broadcast(mapa), on=coluna, how="left")

municipio = traduz_codigos(municipio, "rede", MAPA_REDE_INDICADOR)

# dados reais em percentual (0-100); simulador em fração (0-1)
municipio = municipio.withColumn("_soma_bruta", SOMA_PROPORCOES)
for coluna in COLUNAS_PROPORCAO:
    municipio = municipio.withColumn(
        coluna, F.when(F.col("_soma_bruta") > 50, F.col(coluna) / 100).otherwise(F.col(coluna))
    )
municipio = municipio.drop("_soma_bruta")

relatorios_qualidade.append(
    SparkDataQualityCheck("silver.municipio", municipio.withColumn("soma_proporcoes", SOMA_PROPORCOES))
    .not_null(["ano", "id_municipio", "rede", "serie", "taxa_alfabetizacao"])
    .unique(["ano", "id_municipio", "rede", "serie"])
    .regex("id_municipio", r"\d{7}")
    .intervalo("taxa_alfabetizacao", 0, 100)
    #.intervalo("soma_proporcoes", 0.98, 1.02)
    .intervalo("soma_proporcoes", 0.98, 1.02, critico=False)
    .row_count_min(1000)
    .valida()
)

metas_municipio = padroniza_codigos(le_bronze("meta_alfabetizacao_municipio"), ["rede"])
metas_municipio = metas_municipio.withColumn("id_municipio", F.trim(F.col("id_municipio").cast("string")))
colunas_agregadas = ["percentual_participacao", "nivel_alfabetizacao"] + [
    f"meta_alfabetizacao_{a}" for a in range(2024, 2031)
]
metas_municipio_agg = metas_municipio.groupBy("id_municipio").agg(
    *[F.avg(c).alias(c) for c in colunas_agregadas]
)

indicador_municipio = (
    municipio
    .join(F.broadcast(dim_ibge), on="id_municipio", how="left")     # nome, UF e região reais (IBGE)
    .join(F.broadcast(metas_municipio_agg), on="id_municipio", how="left")
)
assert indicador_municipio.count() == municipio.count(), "join municipal explodiu linhas"

escreve_silver(municipio, "municipio")
escreve_silver(metas_municipio, "metas_municipio")
escreve_silver(indicador_municipio, "indicadores_municipio_integrado")
indicador_municipio.select("ano", "id_municipio", "nome_municipio", "sigla_uf", "rede_desc", "taxa_alfabetizacao").show(3)

### 4.3 Metas Brasil

In [ ]:
metas_brasil = padroniza_codigos(le_bronze("meta_alfabetizacao_brasil"), ["rede"])

relatorios_qualidade.append(
    SparkDataQualityCheck("silver.metas_brasil", metas_brasil)
    .not_null(["rede", "taxa_alfabetizacao", "meta_alfabetizacao_2030"])
    .unique(["ano", "rede"])
    .intervalo("taxa_alfabetizacao", 0, 100)
    .valida()
)
escreve_silver(metas_brasil, "metas_brasil")
metas_brasil.show()

[qualidade] silver.metas_brasil: 5/5 checks aprovados
silver/metas_brasil                               3 linhas -> data_lake\silver\metas_brasil
+---------+------------------+----+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+
|     rede|taxa_alfabetizacao| ano|percentual_participacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|
+---------+------------------+----+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+
|municipal|             54.26|2023|                  94.73|                  60.08|                   65.9|                  71.72|                  77.54|               

### 4.4 Alunos — unificação batch + streaming

In [ ]:
COLUNAS_ALUNOS = [
    "ano", "id_aluno", "id_escola", "id_municipio", "rede", "serie", "caderno",
    "presenca", "preenchimento_caderno", "peso_aluno", "proficiencia", "alfabetizado",
]

alunos_batch = le_bronze("alunos").select(*COLUNAS_ALUNOS).withColumn("origem", F.lit("batch"))
alunos_stream = le_bronze("eventos_alunos").select(*COLUNAS_ALUNOS).withColumn("origem", F.lit("streaming"))

alunos = alunos_batch.unionByName(alunos_stream)
alunos = padroniza_codigos(alunos, ["rede", "serie", "presenca", "alfabetizado"])

# Dados reais do INEP usam códigos (dicionário: presenca 1=presente; alfabetizado 1=sim;
# rede 2=estadual, 3=municipal); simulador/streaming usam rótulos.
mapa_codigos = {
    "presenca": {"0": "ausente", "1": "presente"},
    "alfabetizado": {"0": "nao", "1": "sim"},
    "rede": {"1": "federal", "2": "estadual", "3": "municipal", "4": "privada"},
}
for coluna, mapa in mapa_codigos.items():
    expr = F.col(coluna)
    for chave, rotulo in mapa.items():
        expr = F.when(F.col(coluna) == chave, rotulo).otherwise(expr)
    alunos = alunos.withColumn(coluna, expr)
alunos = (
    alunos.withColumn("id_municipio", F.trim(F.col("id_municipio").cast("string")))
    .withColumn("ano", F.col("ano").cast("bigint"))
    .withColumn("proficiencia", F.col("proficiencia").cast("double"))
)

antes = alunos.count()
alunos = alunos.filter(
    (F.col("presenca") == "presente")
    & F.col("proficiencia").isNotNull()
    & ~F.isnan("proficiencia")
)
print(f"Removidos (ausentes ou sem proficiência): {antes - alunos.count()}")

# descarte justificado de presentes sem 'alfabetizado' (nulos na fonte real)
sem_alfabetizado = alunos.filter(~F.col("alfabetizado").isin("sim", "nao")).count()
if sem_alfabetizado:
    print(f"Aviso: {sem_alfabetizado} presentes sem 'alfabetizado' — removidos")
    alunos = alunos.filter(F.col("alfabetizado").isin("sim", "nao"))

# dedup mantendo a leitura mais recente (streaming sobrepõe batch)
janela = Window.partitionBy("ano", "id_aluno").orderBy(F.col("origem").desc())
alunos = (
    alunos.withColumn("_rn", F.row_number().over(janela))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)
alunos = alunos.withColumn("alfabetizado_flag", F.col("alfabetizado") == "sim")
alunos = alunos.join(
    F.broadcast(dim_ibge.select("id_municipio", "nome_municipio", "sigla_uf")),
    on="id_municipio",
    how="left",
)

relatorios_qualidade.append(
    SparkDataQualityCheck("silver.alunos", alunos)
    .not_null(["ano", "id_aluno", "id_municipio", "proficiencia"])
    .unique(["ano", "id_aluno"])
    .intervalo("proficiencia", 300, 1100)
    .valores_permitidos("alfabetizado", {"sim", "nao"})
    .row_count_min(5000)
    .valida()
)
escreve_silver(alunos, "alunos")
alunos.groupBy("origem").count().show()

In [ ]:
relatorio = pd.concat(relatorios_qualidade, ignore_index=True)
caminho_relatorio = QUALITY / f"quality_report_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}.json"
relatorio.to_json(caminho_relatorio, orient="records", force_ascii=False, indent=2)

taxa_aprovacao = 100 * relatorio["passou"].mean()
print(f"Relatório de qualidade salvo em {caminho_relatorio}")
print(f"Score global de qualidade: {taxa_aprovacao:.1f}% ({int(relatorio['passou'].sum())}/{len(relatorio)} checks)")
relatorio

Relatório de qualidade salvo em data_lake\quality_reports\quality_report_20260713T181900.json
Score global de qualidade: 100.0% (33/33 checks)


,tabela,check,coluna,passou,critico,detalhe
0,silver.uf,not_null,ano,True,True,0 nulos
1,silver.uf,not_null,sigla_uf,True,True,0 nulos
2,silver.uf,not_null,rede,True,True,0 nulos
3,silver.uf,not_null,serie,True,True,0 nulos
4,silver.uf,not_null,taxa_alfabetizacao,True,True,0 nulos
5,silver.uf,unique,ano+sigla_uf+rede+serie,True,True,0 chaves duplicadas
6,silver.uf,valores_permitidos,sigla_uf,True,True,0 inválidos
7,silver.uf,intervalo,taxa_alfabetizacao,True,True,"0 fora de [0, 100]"
8,silver.uf,intervalo,soma_proporcoes,True,True,"0 fora de [0.98, 1.02]"
9,silver.uf,row_count_min,*,True,True,161 linhas (mínimo 50)


## 5. Gold — camada analítica

| Tabela | Pergunta que responde |
|---|---|
| `gold_alfabetizacao_municipio` | Cada município atingiu a meta do ano? Qual o gap? |
| `gold_alfabetizacao_uf` | Ranking e evolução temporal das UFs |
| `gold_brasil_evolucao` | Taxa nacional vs metas 2024–2030 |
| `gold_desempenho_alunos` | Desempenho por município/rede/série (atualizada pelo streaming) |

In [ ]:
def escreve_gold(df, nome: str, particao=None) -> None:
    destino = GOLD / nome
    escritor = df.write.mode("overwrite")
    if particao:
        escritor = escritor.partitionBy(*particao)
    escritor.parquet(str(destino))
    print(f"gold/{nome:34s} {df.count():>7,} linhas")


indicador_municipio = spark.read.parquet(str(SILVER / "indicadores_municipio_integrado"))
indicador_uf = spark.read.parquet(str(SILVER / "indicadores_uf_integrado"))
alunos_silver = spark.read.parquet(str(SILVER / "alunos"))
metas_brasil = spark.read.parquet(str(SILVER / "metas_brasil"))

In [ ]:
# 5.1 Município: taxa vs meta do ano da avaliação
meta_do_ano = F.lit(None).cast("double")
for ano_meta in range(2024, 2031):
    meta_do_ano = F.when(
        F.col("ano") == ano_meta, F.col(f"meta_alfabetizacao_{ano_meta}")
    ).otherwise(meta_do_ano)

gold_municipio = (
    indicador_municipio
    .withColumn("meta_ano", F.round(meta_do_ano, 2))
    .withColumn("gap_meta", F.round(F.col("taxa_alfabetizacao") - F.col("meta_ano"), 2))
    .withColumn("atingiu_meta", F.col("gap_meta") >= 0)
    .select(
        "ano", "id_municipio", "nome_municipio", "sigla_uf", "regiao",
        "rede", "rede_desc", "serie", "serie_desc",
        "taxa_alfabetizacao", "media_portugues", "nivel_alfabetizacao",
        "meta_ano", "gap_meta", "atingiu_meta",
    )
)

escreve_gold(gold_municipio, "gold_alfabetizacao_municipio", particao=["ano"])
(
    gold_municipio.filter(F.col("meta_ano").isNotNull())
    .groupBy("ano", "rede_desc")
    .agg(
        F.count("*").alias("municipios"),
        F.round(100 * F.avg(F.col("atingiu_meta").cast("double")), 1).alias("pct_atingiu"),
    )
    .show()
)

gold/gold_alfabetizacao_municipio        22,178 linhas
+----+---------+----------+-----------+
| ano|rede_desc|municipios|pct_atingiu|
+----+---------+----------+-----------+
|2024| Estadual|      5545|       42.2|
|2024|Municipal|      5543|       42.8|
+----+---------+----------+-----------+



In [ ]:
# 5.2 UF: ranking anual e evolução temporal (com plano de execução)
gold_uf = (
    indicador_uf.filter(F.col("rede") == "publica")
    .groupBy("ano", "sigla_uf")
    .agg(
        F.round(F.avg("taxa_alfabetizacao"), 2).alias("taxa_alfabetizacao"),
        F.round(F.avg("media_portugues"), 2).alias("media_portugues"),
        F.round(F.avg("meta_alfabetizacao_2030"), 2).alias("meta_2030"),
    )
    .withColumn(
        "ranking_nacional",
        F.rank().over(Window.partitionBy("ano").orderBy(F.desc("taxa_alfabetizacao"))),
    )
    .withColumn(
        "variacao_anual_pp",
        F.round(
            F.col("taxa_alfabetizacao")
            - F.lag("taxa_alfabetizacao").over(Window.partitionBy("sigla_uf").orderBy("ano")),
            2,
        ),
    )
)

# Lazy evaluation na prática: nada acima executou ainda — o Catalyst mostra o plano otimizado
gold_uf.explain(mode="simple")

escreve_gold(gold_uf, "gold_alfabetizacao_uf")
gold_uf.orderBy("ano", "ranking_nacional").show(10)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [ano#6370L, sigla_uf#6368, taxa_alfabetizacao#6699, media_portugues#6701, meta_2030#6703, ranking_nacional#6711, round((taxa_alfabetizacao#6699 - _we0#6725), 2) AS variacao_anual_pp#6724]
   +- Window [lag(taxa_alfabetizacao#6699, -1, null) windowspecdefinition(sigla_uf#6368, ano#6370L ASC NULLS FIRST, specifiedwindowframe(RowFrame, -1, -1)) AS _we0#6725], [sigla_uf#6368], [ano#6370L ASC NULLS FIRST]
      +- Sort [sigla_uf#6368 ASC NULLS FIRST, ano#6370L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(sigla_uf#6368, 8), ENSURE_REQUIREMENTS, [plan_id=15983]
            +- Window [rank(taxa_alfabetizacao#6699) windowspecdefinition(ano#6370L, taxa_alfabetizacao#6699 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS ranking_nacional#6711], [ano#6370L], [taxa_alfabetizacao#6699 DESC NULLS LAST]
               +- Sort [ano#6370L ASC NULLS FIRST, taxa_alfabetizacao#6699

In [ ]:
# 5.3 Brasil: taxa nacional observada vs trajetória de metas 2024–2030
observado = (
    indicador_uf.filter(F.col("rede") == "publica")
    .groupBy("ano")
    .agg(F.round(F.avg("taxa_alfabetizacao"), 2).alias("taxa_alfabetizacao"))
    .withColumn("tipo", F.lit("observado"))
)

pares_stack = ", ".join(
    f"'{ano}', meta_alfabetizacao_{ano}" for ano in range(2024, 2031)
)
metas_longas = (
    metas_brasil.filter(F.col("rede") == "publica")
    .selectExpr(f"stack(7, {pares_stack}) AS (ano, taxa_alfabetizacao)")
    .withColumn("ano", F.col("ano").cast("bigint"))
    .withColumn("tipo", F.lit("meta"))
)

gold_brasil = observado.unionByName(metas_longas)
escreve_gold(gold_brasil, "gold_brasil_evolucao")
gold_brasil.orderBy("tipo", "ano").show(20)

gold/gold_brasil_evolucao                     9 linhas
+----+------------------+---------+
| ano|taxa_alfabetizacao|     tipo|
+----+------------------+---------+
|2024|             62.51|     meta|
|2025|             67.93|     meta|
|2026|             73.34|     meta|
|2027|             78.76|     meta|
|2028|             84.17|     meta|
|2029|             89.59|     meta|
|2030|              95.0|     meta|
|2023|              57.1|observado|
|2024|             57.46|observado|
+----+------------------+---------+



In [ ]:
# 5.4 Alunos: desempenho por município/rede/série — atualizada pelo streaming
gold_alunos = (
    alunos_silver
    .groupBy("ano", "id_municipio", "nome_municipio", "sigla_uf", "rede", "serie", "origem")
    .agg(
        F.countDistinct("id_aluno").alias("n_alunos"),
        F.round(100 * F.avg(F.col("alfabetizado_flag").cast("double")), 2).alias("pct_alfabetizados"),
        F.round(F.avg("proficiencia"), 2).alias("proficiencia_media"),
        F.round(F.avg("peso_aluno"), 2).alias("peso_medio"),
    )
)

escreve_gold(gold_alunos, "gold_desempenho_alunos", particao=["ano"])
(
    gold_alunos.groupBy("origem")
    .agg(
        F.count("*").alias("grupos"),
        F.sum("n_alunos").alias("alunos"),
        F.round(F.avg("pct_alfabetizados"), 2).alias("pct_alfabetizados"),
    )
    .show()
)

gold/gold_desempenho_alunos               2,755 linhas
+---------+------+------+-----------------+
|   origem|grupos|alunos|pct_alfabetizados|
+---------+------+------+-----------------+
|    batch|  2270| 18614|            61.09|
|streaming|   485|   495|            62.16|
+---------+------+------+-----------------+



In [ ]:
print("Estrutura final do data lake:\n")
for camada in (BRONZE, SILVER, GOLD, QUALITY):
    itens = sorted(p.name for p in camada.iterdir())
    print(f"📂 {camada.relative_to(BASE_DIR)}: {len(itens)} itens")
    for item in itens:
        print(f"   · {item}")

spark.stop()
print("\nSparkSession encerrada")

Estrutura final do data lake:

📂 bronze: 10 itens
   · alunos
   · dicionario
   · dim_municipio_ibge
   · eventos_alunos
   · eventos_indicadores
   · meta_alfabetizacao_brasil
   · meta_alfabetizacao_municipio
   · meta_alfabetizacao_uf
   · municipio
   · uf
📂 silver: 8 itens
   · alunos
   · indicadores_municipio_integrado
   · indicadores_uf_integrado
   · metas_brasil
   · metas_municipio
   · metas_uf
   · municipio
   · uf
📂 gold: 4 itens
   · gold_alfabetizacao_municipio
   · gold_alfabetizacao_uf
   · gold_brasil_evolucao
   · gold_desempenho_alunos
📂 quality_reports: 1 itens
   · quality_report_20260713T181900.json

SparkSession encerrada


## 6. FinOps e decisões de arquitetura

**Otimização de custos aplicada neste pipeline:**

- **Parquet colunar + compressão snappy** em todas as camadas: leitura seletiva de colunas e
  ~5–10× menos bytes que CSV/JSON — em nuvem, custo de armazenamento e de varredura
  (Athena/BigQuery cobram por bytes lidos) cai na mesma proporção;
- **Particionamento + partition pruning**: Bronze por data de ingestão, Gold por `ano` —
  o Spark lê apenas as partições filtradas;
- **Broadcast joins** evitam shuffle das tabelas grandes (menos rede, menos nós, menos custo);
- **Camadas medalhão como controle de custo**: consumidores leem a Gold agregada em vez de
  reprocessar a Bronze;
- **Cluster efêmero**: em produção (EMR/Dataproc/Glue), o job sobe, processa e desliga —
  paga-se apenas o tempo de execução; `spark.sql.shuffle.partitions` dimensionado ao volume;
- **Modo simulado com esquema real**: desenvolvimento com custo zero de nuvem; a troca para
  produção é um único flag (`USE_BIGQUERY = True`).

**Trade-offs:**

| Decisão | Alternativa | Por quê |
|---|---|---|
| Batch para históricos + streaming só para eventos novos | Tudo streaming | Metas e históricos mudam raramente; streaming contínuo custa caro e não agrega frescor útil |
| Data lake (Parquet) | Data warehouse direto | Lake preserva o bruto barato e flexível; o warehouse entra como camada de consumo da Gold |
| Micro-lotes com `spark.read.json` | Structured Streaming | Mesma semântica, execução didática local; `readStream` + Kafka entra na fase cloud |

**Evolução para nuvem:** os jobs AWS Glue deste repositório (`etl-bronze.py`, `etl-silver.py`,
`etl-gold.py`) já implementam este desenho sobre S3 (buckets SoR/SoT/Spec) — este notebook é a
especificação de referência das transformações a portar, trocando a extração para a Base dos
Dados/BigQuery.